# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [2]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links
from openai import OpenAI 

In [3]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [6]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

MODEL = 'gpt-5-nano'
openai = OpenAI()

#system prompt
system_prompt = """You are a helpful tutor.
When given a code snippet, explain:
1. What code does overall
2. Each part step by step
3. Any important concepts used
Keep explanation clear and beginner-friendly"""

#the tool
def explain_code(code_snippet):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please explain this code:\n\n```\n{code_snippet}\n```"}
    ]

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

#test it with the question from the notebook
code_to_explain = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""
explain_code(code_to_explain)      


Here’s what the code does and how it works, step by step.

What the code does overall
- It defines a simple Python function named fibonacci that computes the nth Fibonacci number using naive recursion.
- The function uses the standard Fibonacci base cases F(0) = 0 and F(1) = 1, and for n >= 2 it uses the relation F(n) = F(n-1) + F(n-2).

Step-by-step explanation
1) def fibonacci(n):
- This line declares a function named fibonacci with one parameter n. You call it like fibonacci(5).

2) if n <= 1:
   return n
- This is the base case. If n is 0 or 1 (or any negative number, due to the <=, see edge notes), the function returns n directly.
- This means:
  - fibonacci(0) returns 0
  - fibonacci(1) returns 1
- The base case stops the recursion from continuing forever.

3) return fibonacci(n-1) + fibonacci(n-2)
- This is the recursive case. For n >= 2, the function calls itself twice:
  - once with n-1
  - once with n-2
- It then adds those two results to produce fibonacci(n).

How the recursion unfolds (example)
- fibonacci(4) = fibonacci(3) + fibonacci(2)
  - fibonacci(3) = fibonacci(2) + fibonacci(1)
    - fibonacci(2) = fibonacci(1) + fibonacci(0)
      - fibonacci(1) = 1
      - fibonacci(0) = 0
    - So fibonacci(2) = 1 + 0 = 1
  - fibonacci(3) = 1 + 1 = 2
  - fibonacci(2) = 1 (as above)
- fibonacci(4) = 2 + 1 = 3
So fibonacci(4) returns 3.

Important concepts used
- Recursion
  - The function solves a problem by calling itself with smaller inputs.
  - Base case prevents infinite recursion; recursive case reduces n.
- The Fibonacci sequence
  - F(0) = 0, F(1) = 1, and F(n) = F(n-1) + F(n-2) for n >= 2.
- Time and space characteristics of naive recursion
  - Time complexity is exponential (roughly O(2^n)) because it recomputes the same values many times.
  - Space complexity is O(n) due to the depth of the call stack (worst-case when computing F(n)).
- Edge cases and potential issues
  - Negative inputs: because the base case is n <= 1, fibonacci(-3) would return -3, which isn’t a valid Fibonacci number. In typical use you’d want to restrict n to non-negative integers and raise an error for invalid input.
  - Recursion depth limit: Python has a recursion depth limit (around 1000 by default). For large n, this can cause a RecursionError.
- Python specifics
  - Python integers are arbitrary-precision, so there’s no overflow for large Fibonacci numbers (other than performance limits).

Possible improvements (optional, not in the original code)
- Use memoization to avoid recomputing values (dramatically speeds up).
  - Example idea: decorate the function with functools.lru_cache or maintain a cache.
- Use an iterative approach (loop) or dynamic programming to compute F(n) in O(n) time and O(1) or O(n) space.
- Add input validation to ensure n is a non-negative integer and raise a clear error for invalid inputs.

Usage examples
- fibonacci(5) returns 5
- fibonacci(6) returns 8
- fibonacci(0) returns 0
- fibonacci(1) returns 1

If you’d like, I can show a version with memoization or an iterative version for comparison.

In [9]:
# Get gpt-4o-mini to answer, with streaming

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

MODEL = MODEL_GPT
openai = OpenAI()

#system prompt
system_prompt = """You are a helpful tutor.
When given a code snippet, explain:
1. What code does overall
2. Each part step by step
3. Any important concepts used
Keep explanation clear and beginner-friendly"""

#the tool
def explain_code(code_snippet):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please explain this code:\n\n```\n{code_snippet}\n```"}
    ]

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

#test it with the question from the notebook
code_to_explain = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""
explain_code(code_to_explain)   


Sure! Let's break down this code step-by-step.

### Overall Purpose

The code defines a function `fibonacci(n)` that computes the nth Fibonacci number. The Fibonacci sequence is a series of numbers where each number (after the first two) is the sum of the two preceding ones. The sequence typically starts with 0 and 1, so the sequence looks like this: 

- 0 (fibonacci(0))
- 1 (fibonacci(1))
- 1 (fibonacci(2) = fibonacci(1) + fibonacci(0))
- 2 (fibonacci(3) = fibonacci(2) + fibonacci(1))
- 3 (fibonacci(4) = fibonacci(3) + fibonacci(2))
- 5 (fibonacci(5) = fibonacci(4) + fibonacci(3))
- and so on...

### Step-by-Step Explanation

1. **Function Definition**:
   ```python
   def fibonacci(n):
   ```
   - This line defines a function named `fibonacci`. It takes one argument `n`, which is expected to be a non-negative integer representing the position in the Fibonacci sequence.

2. **Base Condition**:
   ```python
   if n <= 1:
       return n
   ```
   - Here, we check if `n` is less than or equal to 1. If it is (i.e., if `n` is 0 or 1), the function returns `n`. This is called a "base case" and prevents the function from calling itself infinitely. 
     - For `fibonacci(0)`, it returns `0`.
     - For `fibonacci(1)`, it returns `1`.

3. **Recursive Case**:
   ```python
   return fibonacci(n-1) + fibonacci(n-2)
   ```
   - This line handles the cases where `n` is greater than 1. It calls the function recursively twice:
     - `fibonacci(n-1)` computes the (n-1)th Fibonacci number.
     - `fibonacci(n-2)` computes the (n-2)th Fibonacci number.
   - The results of these two calls are added together. This is how the Fibonacci sequence is built up: each number is the sum of the two previous numbers.

### Important Concepts Used

1. **Recursion**:
   - The function calls itself to compute smaller values until it reaches the base case. Recursion is a powerful programming technique that allows you to solve complex problems by breaking them down into simpler subproblems.

2. **Base Case**:
   - The condition that stops the recursion (in this case, when `n` is 0 or 1). Without a base case, the function would continue to call itself forever, leading to a stack overflow error.

3. **Fibonacci Sequence**:
   - A fundamental concept in mathematics and computer science, useful in various applications like algorithms, data structures, and more.

### Summary

In summary, the `fibonacci(n)` function computes the nth Fibonacci number using a recursive approach. It simplifies the problem by using two recursive calls for each n greater than 1 and has a clear stopping point with the base condition for n being 0 or 1.

In [10]:
# Get Llama 3.2 to answer
ollama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'
)

def explain_code_llama(code_snippet):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please explain this code:\n\n```\n{code_snippet}\n```"}
    ]
    stream = ollama_client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

explain_code_llama(code_to_explain)

I'd be happy to break down this code.

**What does the overall code do?**

This is a recursive function that calculates the `n`th Fibonacci number. The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones: 0, 1, 1, 2, 3, 5, and so on.

**Let's break down the code step by step:**

### Function Definition

```python
def fibonacci(n):
```

This line defines a new function called `fibonacci` that takes one argument, `n`.

### Base Case

```python
if n <= 1:
    return n
```

This is the base case of our recursion. The Fibonacci sequence starts with two special numbers: 0 and 1. To calculate the Fibonacci number for a single position in the sequence (like the number at a specific spot), we just need to know the number before it if `n` is less than or equal to 1.

### Recursive Step

```python
return fibonacci(n-1) + fibonacci(n-2)
```

This is where our recursion truly happens. To calculate the Fibonacci number at position `n`, we actually have to:

1. Calculate the number just before it (`fibonacci(n-1)`).
2. Add the number before that one (`fibonacci(n-2)`).

So, with a recursive step like this (calling `fibonacci` on other values of `n`) and our base case, we can compute Fibonacci numbers for much larger values than we could manually!

### Why does it work?

By using recursion and these two specific rules:

* **Base Case**: We have a clear end point to stop the recursion (i.e., when `n` is less than or equal to 1).
* **Recursive Step**: When we need to find Fibonacci numbers for values "close" in the number line, like `n`, we can break it down into simpler computations involving nearby `n`s!

This process effectively unfolds a way to build up the sequence of Fibonacci numbers through repeated recursive calls (one step backwards at a time!)

Note that the code could grow inefficiently if asked to compute very large Fibonacci numbers recursively because it involves re-computing many values already computed in intermediate steps. If you need larger Fibonacci numbers, consider an alternative implementation using iteration or memoization.

Do these explanations make sense?